In [1]:
# Install the core frameworks
!pip install langgraph langchain

# Install the PDF and Text processing tools
!pip install pypdf2 sentence-transformers scikit-learn

# Install the Privacy & NER tools
!pip install presidio-analyzer presidio-anonymizer spacy

# Download the English NLP model for spaCy/Presidio
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 47.0.0 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 47.0.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.0 MB/s eta 0:00:00


In [1]:
!pip install langchain-text-splitters

In [11]:
from langchain_groq import ChatGroq
import os
from getpass import getpass

# Set your Groq API Key
os.environ["GROQ_API_KEY"] = getpass("RealEst")

RealEst··········


In [15]:
from typing import TypedDict, List
import PyPDF2
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# --- THE STATE ---
class GraphState(TypedDict):
    question: str
    context: List[str]
    summary: str
    matches_article: bool
    has_personal_info: bool

# --- NODE 1: PDF LOADER & CHUNKER ---
def load_and_chunk_pdf(state: GraphState):
    print("---LOG: Loading PDF and Creating Chunks---")
    # Make sure you upload a file named 'legal_doc.pdf' to your notebook folder!
    file_path = "/content/legal_doc.pdf"

    try:
        reader = PyPDF2.PdfReader(file_path)
        full_text = ""
        for page in reader.pages:
            full_text += page.extract_text()

        # Recursive splitting keeps related sentences together
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
        chunks = text_splitter.split_text(full_text)

        return {"context": chunks}
    except Exception as e:
        print(f"Error loading PDF: {e}")
        return {"context": ["Error: Could not read file."]}

# --- NODE 2: THE GROQ SUMMARIZER ---
def generate_summary_node(state: GraphState):
    print("---LOG: Generating Summary with Groq---")

    # We use a fast model like llama-3.1-70b-versatile or mixtral-8x7b-32768
    llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

    prompt = ChatPromptTemplate.from_template("""
    Answer the question using ONLY the provided context.
    If the answer is not in the context, say 'Information not found.'

    Context: {context}
    Question: {question}
    """)

    # Combine chunks into one string for the prompt
    context_str = "\n\n".join(state["context"][:5]) # Taking top 5 chunks for now
    chain = prompt | llm

    response = chain.invoke({"context": context_str, "question": state["question"]})

    return {"summary": response.content}

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load the model once (outside the function for efficiency)
model = SentenceTransformer('all-MiniLM-L6-v2')

def vector_audit_node(state):
    print("---AUDITING FOR HALLUCINATIONS---")
    summary = state["summary"]
    chunks = state["context"]

    # 1. Vectorize everything
    summary_vec = model.encode([summary])
    chunk_vecs = model.encode(chunks)

    # 2. Find the best matching chunk (The "Ground Truth")
    similarities = cosine_similarity(summary_vec, chunk_vecs)
    best_chunk_index = np.argmax(similarities)
    best_score = similarities[0][best_chunk_index]

    # 3. Update the State
    # We use a threshold of 0.7. Below that, the AI is likely hallucinating.
    is_match = best_score > 0.7

    return {"matches_article": is_match}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# Initialize engines
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def privacy_guard_node(state):
    print("---SCANNING FOR PII---")
    text_to_check = state["summary"]

    # 1. Analyze for common entities
    results = analyzer.analyze(text=text_to_check, entities=["PHONE_NUMBER", "PERSON", "EMAIL_ADDRESS"], language='en')

    if not results:
        return {"has_personal_info": False}

    # 2. If results exist, anonymize
    anonymized_result = anonymizer.anonymize(
        text=text_to_check,
        analyzer_results=results
    )

    # 3. Update state with redacted summary and the flag
    return {
        "summary": anonymized_result.text,
        "has_personal_info": False
    }

In [7]:
def router_node(state):
    # 1. The Critical Failure: Hallucination
    if not state["matches_article"]:
        return "re_generate_node" # Send back to try again

    # 2. The Success Path
    # Even if PII was found, it's now redacted in state["summary"]
    return "final_output_node"

In [8]:
from langgraph.graph import StateGraph, END

# Initialize the Graph
workflow = StateGraph(GraphState)

# Add your Nodes
workflow.add_node("loader", load_and_chunk_pdf)
workflow.add_node("summarizer", generate_summary_node)
workflow.add_node("auditor", vector_audit_node)
workflow.add_node("privacy", privacy_guard_node)

# Connect them
workflow.set_entry_point("loader")
workflow.add_edge("loader", "summarizer")
workflow.add_edge("summarizer", "auditor")
workflow.add_edge("auditor", "privacy")

# The Router Edge
workflow.add_conditional_edges(
    "privacy",
    router_node,
    {
        "re_generate_node": "summarizer", # Back to start
        "final_output_node": END           # Done!
    }
)

app = workflow.compile()

In [17]:
# --- THE SMOKE TEST ---
# 1. Initialize State
initial_state = {
    "question": "Who is the current President of Lebanon?",
    "context": [],
    "summary": "",
    "matches_article": False,
    "has_personal_info": False
}

# 2. Run Node 1 (Loader)
state_after_load = load_and_chunk_pdf(initial_state)

# 3. Run Node 2 (Groq Summarizer)
# We update the state with the context we just loaded
initial_state.update(state_after_load)
final_state = generate_summary_node(initial_state)

print("\n--- FINAL RESULT FROM GROQ ---")
print(final_state["summary"])

---LOG: Loading PDF and Creating Chunks---
---LOG: Generating Summary with Groq---

--- FINAL RESULT FROM GROQ ---
Information not found.
